# 04 · Deployment (FastAPI + adapter từ Drive)
Giai đoạn 4: phục vụ model qua API `/extract-cccd/`.

Base **Qwen3-VL-8B** (local) load 4-bit + LoRA adapter **đọc trực tiếp từ Google Drive**, dùng ngrok để lộ public URL.

## 1. Mount Drive + trỏ BASE_MODEL (local) & ADAPTER_DIR vào Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
PROJECT_DRIVE = Path('/content/drive/MyDrive/cccd_project')
os.environ['BASE_MODEL']  = str(PROJECT_DRIVE / 'models' / 'Qwen3-VL-8B-Instruct')
os.environ['ADAPTER_DIR'] = str(PROJECT_DRIVE / 'checkpoints' / 'qwen3vl-cccd-lora')
print('BASE_MODEL  =', os.environ['BASE_MODEL'])
print('ADAPTER_DIR =', os.environ['ADAPTER_DIR'])

## 2. Cài thư viện phục vụ (transformers từ source cho Qwen3-VL)

In [ ]:
!cp -r '/content/drive/MyDrive/cccd_project/code' /content/cccd 2>/dev/null || echo 'Đặt source vào /content/cccd'
%cd /content/cccd
!pip -q install git+https://github.com/huggingface/transformers
!pip -q install qwen-vl-utils accelerate peft bitsandbytes Pillow fastapi uvicorn python-multipart pyngrok

## 3. Khởi động FastAPI (nền) + ngrok public URL

In [ ]:
import subprocess, time
from pyngrok import ngrok

# ngrok.set_auth_token('YOUR_TOKEN')  # nếu cần token riêng
proc = subprocess.Popen(['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(60)  # chờ model 8B load 4-bit từ Drive
public_url = ngrok.connect(8000)
print('API public URL:', public_url)
print('Swagger UI    :', f'{public_url}/docs')

## 4. Test thử 1 ảnh

In [ ]:
import requests, glob
sample = glob.glob(str(PROJECT_DRIVE / 'data' / 'raw' / '*'))[0]
with open(sample, 'rb') as f:
    r = requests.post(f'{public_url}/extract-cccd/', files={'file': f}, params={'side': 'auto'})
import json; print(json.dumps(r.json(), ensure_ascii=False, indent=2))